In [3]:
from read_model_runs import read_model_runs

# Ler dados completos
question_long_df = \
    read_model_runs('../../data/processed/model-runs')


In [4]:
from read_model_runs import filter_complete_questions

models = ['gemma-3-27b-it', 'gemma-3-12b-it', 'gemma-3-4b-it']
firacs = ['FILA_', 'FIR__', 'FI___', 'FIL__', '_____', 'unstructured']
question_long_df, question_wide_df, model_order, firac_order = filter_complete_questions(question_long_df, models, firacs)

print('shape:', question_long_df.shape)
print('# unique questions:', question_long_df['question_id'].nunique())
print("model order:", model_order)
print("firac order:", firac_order)


shape: (41850, 32)
# unique questions: 2325
model order: ['gemma-3-4b-it', 'gemma-3-12b-it', 'gemma-3-27b-it']
firac order: ['_____', 'unstructured', 'FIL__', 'FI___', 'FIR__', 'FILA_']


In [6]:
import json

def count_source(row):
    rule_dict = json.loads(row["Rule"])
    return sum(1 for v in rule_dict.values() if v != [])

rule = """{"constituicao_federal": [], "codigo_penal": [] } """
row = {}
row['Rule'] = rule
assert count_source(row) == 0

rule = """{"constituicao_federal": [], "codigo_penal": [], "codigo_processo_penal": [], "clt": [], "codigo_civil": [], "codigo_processo_civil": [], "codigo_direito_consumidor": [], "estatuto_crianca_adolescente": [], "sumula_stj": [], "estatuto_oab": [{"numero": "Art. 31, \u00a71\u00ba do EAOAB", "conteudo": "O advogado possui liberdade e independ\u00eancia em sua atua\u00e7\u00e3o."}, {"numero": "Art. 6\u00ba do EAOAB", "conteudo": "N\u00e3o h\u00e1 qualquer hierarquia entre advogados, ju\u00edzes e membros do Minist\u00e9rio P\u00fablico."}], "codigo_tributario_nacional": [], "lei_federal": [{"numero": "Par\u00e1grafo \u00fanico do Art. 4\u00ba do C\u00f3digo de \u00c9tica e Disciplina da Advocacia", "conteudo": "O advogado n\u00e3o \u00e9 obrigado a atuar contra seu entendimento j\u00e1 formulado em parecer anterior."}], "doutrina": []}"""
row = {}
row['Rule'] = rule
assert count_source(row) == 2

In [10]:
import pandas as pd

# Carrega o dataframe base (um registro por questão)
exam_df = pd.read_csv("../../data/processed/oab_with_firac_portuguese_shuffle.csv")

# Calcula rule_source_count no nível da questão
exam_df["rule_source_count"] = exam_df.apply(count_source, axis=1)

exam_df[["question_id", "rule_count", "rule_source_count"]].to_csv("../../data/processed/question_level_stats.csv", index=False)


In [ ]:
import pandas as pd

# --------------------------------------------------
# 1) Média de acurácia por question_id × firac
# --------------------------------------------------
mean_by_firac = (
    question_long_df
    .groupby(["question_id", "firac"], as_index=False)
    .agg(
        mean_accuracy_firac=("is_correct", "mean"),
        n_obs_firac=("is_correct", "size")
    )
)

# --------------------------------------------------
# 2) Média geral (todos os FIRACs) por question_id
# --------------------------------------------------
mean_overall = (
    question_long_df
    .groupby("question_id", as_index=False)
    .agg(
        mean_accuracy_all=("is_correct", "mean"),
        n_obs_all=("is_correct", "size")
    )
)

# --------------------------------------------------
# 3) Merge
# --------------------------------------------------
df = mean_by_firac.merge(
    mean_overall,
    on="question_id",
    how="left"
)

# --------------------------------------------------
# 4) Média dos "outros" FIRACs
#    (remove contribuição do próprio nível)
# --------------------------------------------------
df["mean_accuracy_other_firac"] = (
    (df["mean_accuracy_all"] * df["n_obs_all"]
     - df["mean_accuracy_firac"] * df["n_obs_firac"])
    / (df["n_obs_all"] - df["n_obs_firac"])
)

# --------------------------------------------------
# 5) Diferença (delta FIRAC)
# --------------------------------------------------
df["delta_accuracy_firac"] = (
    df["mean_accuracy_firac"]
    - df["mean_accuracy_other_firac"]
)

# Resultado final
delta_firac_df = df[
    [
        "question_id",
        "firac",
        "mean_accuracy_firac",
        "mean_accuracy_other_firac",
        "delta_accuracy_firac",
    ]
]

import pandas as pd

delta_firac_wide_df = (
    delta_firac_df
    .pivot(
        index="question_id",
        columns="firac",
        values="delta_accuracy_firac"
    )
    .reset_index()
)

delta_firac_wide_df.to_csv("../../data/processed/delta_firac.csv", index=False)

delta_firac_wide_df.head()

firac,question_id,FILA_,FIL__,FIR__,FI___,_____,unstructured
0,oab-1.pdf-002,0.133333,0.133333,0.133333,0.133333,-0.666667,0.133333
1,oab-1.pdf-003,0.666667,-0.133333,0.666667,-0.133333,-0.533333,-0.533333
2,oab-1.pdf-004,0.266667,-0.133333,0.266667,-0.133333,-0.533333,0.266667
3,oab-1.pdf-005,-0.400000,0.400000,-0.800000,0.400000,0.400000,0.000000
4,oab-1.pdf-006,0.133333,-0.266667,0.133333,-0.266667,0.133333,0.133333
